In [2]:
import requests
from bs4 import BeautifulSoup
import re

COOKIES_FILE = r"C:\Users\JOAO PC\Documents\MBA---Data-Engineering\cookies.txt"
BASE_ON1 = "https://on1.fiap.com.br"
BASE_PROG = BASE_ON1 + "/programas"
ARQUIVOS_URL = BASE_PROG + "/login/alunos_2004/apostilas_2007/_arquivos.asp"
ARQUIVOS_PASTA_URL = BASE_PROG + "/login/alunos_2004/apostilas_2007/_arquivosPasta.asp"

session = requests.Session()
with open(COOKIES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) < 7:
            continue
        domain, _, path, secure, expires, name, value = parts[:7]
        session.cookies.set(name.strip(), value.strip(), domain=domain.strip(), path=path.strip())

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": BASE_PROG + "/login/alunos_2004/apostilas_2007/default.asp",
    "Origin": BASE_ON1,
})

# Testa Graph Databases
params = {
    "intCurso": "209", "intCursoAno": "2025",
    "strTurma": "29ABD", "intTurmaAno": "2025",
    "intCodDisciplina": "4599", "intCodProfessor": "222",
    "intAno": "", "local": "div_dummy"
}

resp = session.post(ARQUIVOS_URL, data=params)
html = resp.text
if "|" in html[:100]:
    html = html[html.index("|") + 1:]

print(html[:3000])



<div class="i-apostilas-item">
    Não há nenhuma apostila cadastrada.
</div>

<div id="curso209_2025turma29ABD_2025disciplina4599_222_pasta154" class="i-apostilas-item i-lista-apostila-turma-on">
	<span onclick="javascript:fPasta(this.getElementsByTagName('img')[0],this.getElementsByTagName('img')[1],'curso209_2025turma29ABD_2025disciplina4599_222_pasta154all','','intCurso=209&intCursoAno=2025&strTurma=29ABD&intTurmaAno=2025&intCodDisciplina=4599&intCodProfessor=222&intCodPasta=154&intAno=')" class="i-apostilas-label">
        <div class="i-apostilas-label-column">
		    <img src="login/alunos_2004/apostilas_2007/mais.svg" width="9" height="9" />
        </div>

        <div class="i-apostilas-label-column">
		    <svg class="i-apostilas-label-icon" viewBox="0 0 512 512"><use xmlns:xlink="http://www.w3.org/1999/xlink" xlink:href="/dist/desktop/svg/sprites.svg#icon-apostilas-folder"></use></svg>
        </div>

		<div class="i-apostilas-label-column">
            <span class="i-apost

In [4]:
from bs4 import BeautifulSoup
from urllib.parse import unquote
from pathlib import Path
import re

resp = session.post(ARQUIVOS_PASTA_URL, data={
    "intCurso": "209", "intCursoAno": "2025",
    "strTurma": "29ABD", "intTurmaAno": "2025",
    "intCodDisciplina": "4599", "intCodProfessor": "222",
    "intCodPasta": "154", "intAno": "", "local": "div_dummy"
})

html = resp.text
if "|" in html[:100]:
    html = html[html.index("|") + 1:]

# Formato 2 — download.php nos comentários
matches = re.findall(r'download\.php\?file=([^&"\'\\s]+)&(?:amp;)?origem=asp', html)
print("Formato 2 (download.php):", matches)

# Formato 3 — /updown/
matches3 = re.findall(r'href="/updown/([^"]+)"', html)
print("Formato 3 (/updown/):", matches3)

Formato 2 (download.php): []
Formato 3 (/updown/): ['upload_fiap/alunos/apostilas/29ABD - Apresentação.pdf', 'upload_fiap/alunos/apostilas/29ABD - Aula_01_v2.pdf', 'upload_fiap/alunos/apostilas/Hands_on_Grafos_aula_1_v2(2).pdf', 'upload_fiap/alunos/apostilas/29ABD - Aula_02_v3.pdf', 'upload_fiap/alunos/apostilas/Hands_on_Grafos_aula_2_v1(2).pdf', 'upload_fiap/alunos/apostilas/29ABD - Aula_03_v7.pdf', 'upload_fiap/alunos/apostilas/29ABD - Aula_03_v7.pdf']


In [5]:
BASE_ON1 = "https://on1.fiap.com.br"

arquivos = []
vistos = set()

# Formato 3
for url_m in re.finditer(r'href="/updown/([^"]+)"', html):
    file_path = unquote(url_m.group(1))
    nome = Path(file_path).name
    if nome in vistos:
        continue
    vistos.add(nome)
    url = BASE_ON1 + "/updown/" + url_m.group(1)
    arquivos.append({"nome": nome, "url": url})

for a in arquivos:
    print(a["nome"], "->", a["url"])

29ABD - Apresentação.pdf -> https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/29ABD - Apresentação.pdf
29ABD - Aula_01_v2.pdf -> https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/29ABD - Aula_01_v2.pdf
Hands_on_Grafos_aula_1_v2(2).pdf -> https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/Hands_on_Grafos_aula_1_v2(2).pdf
29ABD - Aula_02_v3.pdf -> https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/29ABD - Aula_02_v3.pdf
Hands_on_Grafos_aula_2_v1(2).pdf -> https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/Hands_on_Grafos_aula_2_v1(2).pdf
29ABD - Aula_03_v7.pdf -> https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/29ABD - Aula_03_v7.pdf


In [8]:
from urllib.parse import quote

# Testa download direto sem redirect
url = "https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/29ABD - Apresentação.pdf"
url_encoded = "https://on1.fiap.com.br/updown/" + quote("upload_fiap/alunos/apostilas/29ABD - Apresentação.pdf", safe="/")

# Sem seguir redirect
probe = session.get(url_encoded, allow_redirects=False)
print("Status probe:", probe.status_code)
print("Location:", probe.headers.get("Location", "N/A")[:100])

# Seguindo redirect
resp = session.get(url_encoded, allow_redirects=True)
print("Status final:", resp.status_code)
print("URL final:", resp.url[:100])

Status probe: 200
Location: N/A
Status final: 200
URL final: https://on1.fiap.com.br/updown/upload_fiap/alunos/apostilas/29ABD%20-%20Apresenta%C3%A7%C3%A3o.pdf


In [10]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import unquote, quote
from pathlib import Path
import re

COOKIES_FILE = r"C:\Users\JOAO PC\Documents\MBA---Data-Engineering\cookies.txt"
BASE_ON1 = "https://on1.fiap.com.br"
BASE_PROG = BASE_ON1 + "/programas"
ARQUIVOS_URL = BASE_PROG + "/login/alunos_2004/apostilas_2007/_arquivos.asp"
ARQUIVOS_PASTA_URL = BASE_PROG + "/login/alunos_2004/apostilas_2007/_arquivosPasta.asp"

# Sessão
session = requests.Session()
with open(COOKIES_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) < 7:
            continue
        domain, _, path, secure, expires, name, value = parts[:7]
        session.cookies.set(name.strip(), value.strip(), domain=domain.strip(), path=path.strip())

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": BASE_PROG + "/login/alunos_2004/apostilas_2007/default.asp",
    "Origin": BASE_ON1,
})

def parse_arquivos(html):
    soup = BeautifulSoup(html, "html.parser")
    arquivos = []
    vistos = set()

    # Prioridade 1: /updown/
    for url_m in re.finditer(r'href="/updown/([^"]+)"', html):
        file_path = unquote(url_m.group(1))
        nome = Path(file_path).name
        if nome in vistos:
            continue
        vistos.add(nome)
        url = BASE_ON1 + "/updown/" + quote(url_m.group(1), safe="/")
        arquivos.append({"nome": nome, "url": url})

    # Prioridade 2: download.php
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        if "download.php" not in href or "file=" not in href:
            continue
        m = re.search(r'file=([^&]+)', href)
        if not m:
            continue
        nome = unquote(Path(m.group(1)).name)
        if nome in vistos:
            continue
        vistos.add(nome)
        url = BASE_ON1 + "/programas/" + href.lstrip("./") if not href.startswith("http") else href
        arquivos.append({"nome": nome, "url": url})

    return arquivos

# Testa Graph Databases — subpasta aulas (intCodPasta=154)
params = {
    "intCurso": "209", "intCursoAno": "2025",
    "strTurma": "29ABD", "intTurmaAno": "2025",
    "intCodDisciplina": "4599", "intCodProfessor": "222",
    "intCodPasta": "154", "intAno": "", "local": "div_dummy"
}

resp = session.post(ARQUIVOS_PASTA_URL, data=params)
html = resp.text
if "|" in html[:100]:
    html = html[html.index("|") + 1:]

arquivos = parse_arquivos(html)
print(f"Arquivos encontrados: {len(arquivos)}")
for arq in arquivos:
    r = session.get(arq["url"], stream=True)
    print(f"  {'[OK]' if r.status_code == 200 else '[ERRO ' + str(r.status_code) + ']'} {arq['nome']}")

Arquivos encontrados: 6
  [OK] 29ABD - Apresentação.pdf
  [OK] 29ABD - Aula_01_v2.pdf
  [OK] Hands_on_Grafos_aula_1_v2(2).pdf
  [OK] 29ABD - Aula_02_v3.pdf
  [OK] Hands_on_Grafos_aula_2_v1(2).pdf
  [OK] 29ABD - Aula_03_v7.pdf
